# Ground truth rendering and comparison

Draws every `data/plan*/truth.json` (metres: hall, lobby, wall, doors, booths, tables and chairs, stage, kids, pilasters, fixtures) next to its `layout.png`, and as an overlay on it. `data/plan*/frame.json` gives the pixel scale and origin of each plan.

Output is HD: everything is drawn at `UP` times the pixels of `layout.png` (default 2), and plans with a `source.pdf` (plans 5 and 6) are re-rendered from the PDF vectors at that scale, so the original side is sharp too. Plans without a PDF are upscaled from the raster. PNGs go to `output/truth/<plan>` at full size; the PDF pages are capped at `MAX_PANEL_W` and `MAX_OVERLAY_W` pixels wide and saved as high quality JPEG.

In [ ]:
import json
from pathlib import Path

import cv2
import numpy as np
from IPython.display import Image, display
from PIL import Image as PILImage

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output/truth"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UP = 2                 # draw at UP x the pixels of layout.png
MAX_PANEL_W = 3600     # each panel of a PDF side-by-side page, pixels wide
MAX_OVERLAY_W = 5000   # a PDF overlay page, pixels wide
PDF_QUALITY = 95       # JPEG quality inside the PDF
# plans with a vector source: (clip x0, y0, x1, y1 in PDF points, zoom that made layout.png)
PDF_CLIPS = {"plan5": ((105, 70, 300, 495), 5), "plan6": ((20, 60, 596, 690), 4)}

# BGR colours
HALL = (214, 244, 255)
LOBBY = (252, 238, 222)
BOOTH = {"goods": (150, 215, 150), "fnb": (90, 180, 250), "fit": (230, 170, 200)}
TABLE = (230, 160, 110)
KIDS = (250, 215, 225)
STAGE = (80, 80, 170)
FIXTURE = (190, 190, 190)
CLOSED = (200, 200, 250)
DOOR = (200, 60, 230)


In [ ]:
def load_plan(plan_name):
    truth = json.loads((DATA_DIR / plan_name / "truth.json").read_text())
    frame = json.loads((DATA_DIR / plan_name / "frame.json").read_text())
    return truth, frame


def load_original(plan_name, up):
    """layout.png at up x its size. Plans with a source.pdf are re-rendered from the vectors at that scale."""
    layout = cv2.imread(str(DATA_DIR / plan_name / "layout.png"))
    H, W = layout.shape[:2]
    pdf = DATA_DIR / plan_name / "source.pdf"
    if plan_name in PDF_CLIPS and pdf.exists():
        import pymupdf
        clip, zoom = PDF_CLIPS[plan_name]
        pix = pymupdf.open(pdf)[0].get_pixmap(matrix=pymupdf.Matrix(zoom * up, zoom * up), clip=pymupdf.Rect(*clip), alpha=False)
        img = cv2.cvtColor(np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, 3), cv2.COLOR_RGB2BGR)
        if abs(img.shape[1] - W * up) <= up and abs(img.shape[0] - H * up) <= up:
            return cv2.resize(img, (W * up, H * up), interpolation=cv2.INTER_AREA) if img.shape[:2] != (H * up, W * up) else img
    return cv2.resize(layout, (W * up, H * up), interpolation=cv2.INTER_CUBIC)


class Px:
    """Metres to pixels of the HD canvas: pixel = ((metres - 1) * px_per_m + origin) * up."""

    def __init__(self, frame, up=1):
        self.up = up
        self.s = frame["px_per_m"] * up
        self.ox, self.oy = frame["origin_px"][0] * up, frame["origin_px"][1] * up

    def pt(self, x, y):
        return int(round((x - 1) * self.s + self.ox)), int(round((y - 1) * self.s + self.oy))

    def rect(self, r):
        if "cx" in r:
            return self.pt(r["cx"] - r["w"] / 2, r["cy"] - r["h"] / 2), self.pt(r["cx"] + r["w"] / 2, r["cy"] + r["h"] / 2)
        return self.pt(r["x0"], r["y0"]), self.pt(r["x1"], r["y1"])


def draw_label(img, text, p0, p1, color=(0, 0, 0), up=1, corner=False):
    """Centred label; corner=True writes it small in the top-left corner, so it does not print over the original text."""
    w, h = p1[0] - p0[0], p1[1] - p0[1]
    scale = min(0.65 * up, max(0.2 * up, min(h, w * 0.6) / 38.0))
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, up)
    if tw > max(2, w - 2):
        scale = max(0.18 * up, scale * (w - 2) / tw)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, up)
    if corner:
        scale = min(scale, 0.34 * up)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, up)
        org = (p0[0] + 2 * up, p0[1] + th + 2 * up)
    else:
        org = (p0[0] + max(1, (w - tw) // 2), p0[1] + (h + th) // 2)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2 * up, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, up, cv2.LINE_AA)


def draw_truth(canvas, truth, frame, up=1, filled=True):
    """Draw the metre truth on an HD canvas (up x layout pixels). filled=False draws outlines only, for the overlay."""
    px = Px(frame, up)

    def T(v):
        return max(1, int(round(v * up)))

    def box(r, color, thickness=1):
        p0, p1 = px.rect(r)
        if filled:
            cv2.rectangle(canvas, p0, p1, color, -1)
            cv2.rectangle(canvas, p0, p1, tuple(int(c * 0.5) for c in color), T(1))
        else:
            cv2.rectangle(canvas, p0, p1, color, T(thickness))
        return p0, p1

    if filled:
        cv2.fillPoly(canvas, [np.array([px.pt(*p) for p in truth["hall"]], np.int32)], HALL)
        cv2.fillPoly(canvas, [np.array([px.pt(*p) for p in truth["lobby"]], np.int32)], LOBBY)
    cv2.polylines(canvas, [np.array([px.pt(*p) for p in truth["hall"]], np.int32)], True, (0, 0, 0), T(2))
    cv2.polylines(canvas, [np.array([px.pt(*p) for p in truth["lobby"]], np.int32)], True, (60, 60, 60), T(2))
    for c in truth["closed"]:
        box(c, CLOSED, 2)
    kids = truth["kids"]
    box(kids, KIDS, 2)
    stage = truth["stage"]
    if any(stage[k] for k in ("x0", "x1", "y0", "y1")):
        p0, p1 = box(stage, STAGE, 3)
        draw_label(canvas, "STAGE", p0, p1, color=(255, 255, 255) if filled else (0, 0, 160), up=up, corner=not filled)
    for f in truth["fixtures"]:
        p0, p1 = box(f, FIXTURE, 2)
        draw_label(canvas, f["label"], p0, p1, up=up, corner=not filled)
    for b in truth["booths"]:
        p0, p1 = box(b, BOOTH.get(b["type"], (200, 200, 200)), 2)
        draw_label(canvas, b["id"], p0, p1, up=up, corner=not filled)
    for t in truth["tables"]:
        if t["kind"] == "table":
            long_side, short_side = t["rect"]
            a = np.radians(t["angle"])
            u, v = np.array([np.cos(a), np.sin(a)]), np.array([-np.sin(a), np.cos(a)])
            c0 = np.array([t["cx"], t["cy"]])
            corners = [c0 + sx * long_side / 2 * u + sy * short_side / 2 * v for sx, sy in ((-1, -1), (1, -1), (1, 1), (-1, 1))]
            pts = np.array([px.pt(*p) for p in corners], np.int32)
            if filled:
                cv2.fillPoly(canvas, [pts], (235, 215, 190))
            cv2.polylines(canvas, [pts], True, (160, 90, 30), T(2))
        else:
            box(t, TABLE, 1)
    for p in truth["pilasters"]:
        box(p, (0, 0, 0), 3) if p["kind"] != "block" else box(p, (60, 60, 60), 3)
    for c in truth["columns"]:
        cv2.circle(canvas, px.pt(c["x"], c["y"]), max(2, int(c["d"] / 2 * px.s)), (0, 0, 0), -1 if filled else T(2))
    box(truth["wall"], (0, 0, 0), 3)
    for d in truth["doors"]:
        p0, p1 = box(d, DOOR, 3)
        draw_label(canvas, d["id"][-1] + d["role"][0].upper(), p0, p1, up=up)
    for o in truth["outerDoors"]:
        cv2.line(canvas, px.pt(o["x"], o["y0"]), px.pt(o["x"], o["y1"]), DOOR, T(4))
    for g in truth["gates"]:
        cv2.circle(canvas, px.pt(g["sx"], g["sy"]), T(9), (0, 160, 0), -1)
    e = truth["exitPt"]
    cv2.circle(canvas, px.pt(e["sx"], e["sy"]), T(9), (0, 0, 220), -1)
    return canvas


def render_truth_canvas(frame, truth, w, h, up=1):
    canvas = np.full((h * up, w * up, 3), 255, dtype=np.uint8)
    step = max(2, int(round(frame["px_per_m"] * 5 * up)))   # a grid line every 5 m
    for pos in range(0, max(w, h) * up, step):
        cv2.line(canvas, (pos, 0), (pos, h * up), (228, 228, 228), up)
        cv2.line(canvas, (0, pos), (w * up, pos), (228, 228, 228), up)
    return draw_truth(canvas, truth, frame, up, filled=True)


def render_truth_overlay(orig_hd, truth, frame, up=1):
    return draw_truth(orig_hd.copy(), truth, frame, up, filled=False)


In [ ]:
def fit_width(img, target_w):
    if img.shape[1] <= target_w:
        return img
    return cv2.resize(img, (target_w, int(round(img.shape[0] * target_w / img.shape[1]))), interpolation=cv2.INTER_AREA)


def make_side_by_side(plan_name, orig, rend, summary, max_panel_w=MAX_PANEL_W):
    orig, rend = fit_width(orig, max_panel_w), fit_width(rend, max_panel_w)
    h, w = orig.shape[:2]
    k = max(1.0, w / 1600)                      # header text grows with the page
    bar_h, gap = int(60 * k), int(30 * k)
    combo = np.full((h + bar_h, w * 2 + gap, 3), 245, dtype=np.uint8)
    combo[bar_h:, 0:w] = orig
    combo[bar_h:, w + gap:] = rend
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.9 * k, max(1, int(round(2 * k)))
    cv2.putText(combo, f"Original: {plan_name}/layout.png", (int(20 * k), int(40 * k)), font, scale, (40, 40, 40), thick, cv2.LINE_AA)
    cv2.putText(combo, f"Truth render: {plan_name}/truth.json ({summary})", (w + gap + int(20 * k), int(40 * k)), font, scale, (0, 110, 200), thick, cv2.LINE_AA)
    return combo


def make_overlay_page(plan_name, overlay, summary, max_w=MAX_OVERLAY_W):
    overlay = fit_width(overlay, max_w)
    h, w = overlay.shape[:2]
    k = max(1.0, w / 1600)
    bar_h = int(60 * k)
    page = np.full((h + bar_h, w, 3), 245, dtype=np.uint8)
    page[bar_h:, :] = overlay
    cv2.putText(page, f"Truth overlay on the original: {plan_name} ({summary})", (int(20 * k), int(40 * k)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9 * k, (0, 110, 200), max(1, int(round(2 * k))), cv2.LINE_AA)
    return page


def summarize(truth):
    kinds = {}
    for b in truth["booths"]:
        kinds[b["type"]] = kinds.get(b["type"], 0) + 1
    booths = ", ".join(f"{n} {k}" for k, n in sorted(kinds.items()))
    walls = 1 + sum(1 for p in truth["pilasters"] if p["kind"] == "wall")
    return f"{booths}; {sum(1 for t in truth['tables'] if t['kind'] == 'table')} tables; {sum(1 for t in truth['tables'] if t['kind'] == 'chair')} chairs; {walls} walls; {len(truth['doors'])} doors"


def render_plan(plan_name, display_width=1600, up=UP):
    img_path = DATA_DIR / f"{plan_name}/layout.png"
    if not (DATA_DIR / plan_name / "truth.json").exists() or not (DATA_DIR / plan_name / "frame.json").exists() or not img_path.exists():
        print(f"Skipping {plan_name}: missing truth.json, frame.json or layout.png")
        return None
    truth, frame = load_plan(plan_name)
    H, W = cv2.imread(str(img_path)).shape[:2]
    orig = load_original(plan_name, up)
    summary = summarize(truth)
    print(f"=== {plan_name} Ground Truth ===")
    print(f"layout.png {W} x {H} px, drawn at {W * up} x {H * up} px | {frame['px_per_m']} px/m | world {truth['meta']['world']['w']} x {truth['meta']['world']['h']} m"
          + (" | original re-rendered from source.pdf" if plan_name in PDF_CLIPS else ""))
    print(summary)
    canvas = render_truth_canvas(frame, truth, W, H, up)
    overlay = render_truth_overlay(orig, truth, frame, up)
    sbs = make_side_by_side(plan_name, orig, canvas, summary)
    p_out = OUTPUT_DIR / plan_name
    p_out.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(p_out / "render.png"), canvas)
    cv2.imwrite(str(p_out / "overlay.png"), overlay)
    cv2.imwrite(str(p_out / "side_by_side.png"), sbs)
    if display_width:
        shown = fit_width(sbs, display_width)
        display(Image(data=cv2.imencode(".png", shown)[1].tobytes()))
    return {"plan": plan_name, "summary": summary}


In [ ]:
res1 = render_plan("plan1")

In [ ]:
res2 = render_plan("plan2")

In [ ]:
res3 = render_plan("plan3")

In [ ]:
res5 = render_plan("plan5")

In [ ]:
res6 = render_plan("plan6")

In [ ]:
# Export all ground truth comparisons into a single multi-page PDF
def export_all_pdf(pdf_path="output/truth_comparison.pdf"):
    out_pdf = ROOT / pdf_path
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    plans = [p.parent.name for p in sorted(DATA_DIR.glob("plan*/truth.json")) if (p.parent / "frame.json").exists()]
    pdf_pages = []
    for pname in plans:
        render_plan(pname, display_width=None)
        pdf_pages.append(PILImage.open(OUTPUT_DIR / pname / "side_by_side.png").convert("RGB"))
        truth, frame = load_plan(pname)
        orig = load_original(pname, UP)
        over_page = make_overlay_page(pname, render_truth_overlay(orig, truth, frame, UP), summarize(truth))
        pdf_pages.append(PILImage.fromarray(cv2.cvtColor(over_page, cv2.COLOR_BGR2RGB)))
    if pdf_pages:
        pdf_pages[0].save(out_pdf, save_all=True, append_images=pdf_pages[1:], resolution=200, quality=PDF_QUALITY)
        print(f"Saved {out_pdf} ({len(pdf_pages)} pages, {out_pdf.stat().st_size / (1024 * 1024):.2f} MB)")
        mirror_pdf = OUTPUT_DIR / "truth_comparison.pdf"
        if mirror_pdf.resolve() != out_pdf.resolve():
            import shutil
            shutil.copy2(out_pdf, mirror_pdf)
            print(f"Mirrored to {mirror_pdf}")


export_all_pdf()
